# 05 — Strategy Analysis

Deep-dive into agent behaviour, conviction calibration, and system-level health.

**Topics covered:**
- Agent verdict distribution (BUY / SELL / NO_TRADE / WATCH)
- Conviction vs. actual outcome calibration curve
- Bull vs Bear researcher agreement rate
- Risk manager override frequency
- Intermarket correlation regime impact
- Anti-tilt trigger analysis
- QuantStats HTML tear sheet

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

DB_PATH = "../data/nexus.duckdb"

In [ ]:
conn = duckdb.connect(DB_PATH, read_only=True)
trades = conn.execute("SELECT * FROM trades ORDER BY opened_at").fetchdf()
conn.close()
print(f"Trades: {len(trades)}")

In [ ]:
# Verdict distribution
if not trades.empty and 'side' in trades.columns:
    fig = px.pie(trades, names='side', title='Trade Direction Distribution')
    fig.show()

In [ ]:
# Conviction vs outcome calibration
if not trades.empty and 'conviction' in trades.columns:
    trades['won'] = (trades['pnl_usd'] > 0).astype(int)
    trades['conviction_bin'] = pd.cut(trades['conviction'], bins=np.arange(0, 1.1, 0.1))
    calib = trades.groupby('conviction_bin', observed=True).agg(
        actual_win_rate=('won', 'mean'),
        count=('won', 'count'),
    ).reset_index()
    calib['midpoint'] = calib['conviction_bin'].apply(lambda x: x.mid)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=calib['midpoint'], y=calib['actual_win_rate'], mode='markers+lines', name='Actual Win Rate'))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', line=dict(dash='dash'), name='Perfect Calibration'))
    fig.update_layout(title='Conviction Calibration Curve', xaxis_title='Conviction', yaxis_title='Actual Win Rate')
    fig.show()

In [ ]:
# P&L by session
if not trades.empty and 'session' in trades.columns:
    fig = px.box(trades, x='session', y='pnl_usd', title='P&L Distribution by Session', color='session')
    fig.show()

In [ ]:
# R:R distribution
if not trades.empty and 'rr_ratio' in trades.columns:
    fig = px.histogram(trades, x='rr_ratio', nbins=30, title='Risk:Reward Ratio Distribution', color_discrete_sequence=['#3B82F6'])
    fig.add_vline(x=1.5, line_dash='dash', annotation_text='Min R:R (1.5)')
    fig.show()

In [ ]:
# QuantStats tear sheet
try:
    import quantstats as qs
    if not trades.empty and 'pnl_usd' in trades.columns:
        returns = trades.set_index('closed_at')['pnl_usd'].resample('D').sum().pct_change().dropna()
        qs.reports.html(returns, output='../data/strategy_report.html', title='Nexus Trader Strategy Report')
        print('Report saved to ../data/strategy_report.html')
except ImportError:
    print('Install quantstats: pip install quantstats')